# Python reference 7: classes, dataclasses and protocols

*Author: [Sai Guntupalli](https://github.com/sai-guntupalli) · [sai-guntupalli/data-engineering-handbook](https://github.com/sai-guntupalli/data-engineering-handbook) · CC BY 4.0*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sai-guntupalli/data-engineering-handbook/blob/master/notebooks/topics/python/learning/python-reference-07-classes-and-dataclasses.ipynb)

Instance, class and static methods, properties, inheritance and `super()`; special methods for equality, hashing, ordering and containers; `__slots__`; dataclasses in depth; enums; and abstract base classes versus `Protocol`. Practise it in [Python practice 7: classes, dataclasses and protocols](../practice/python-practice-07-classes-and-dataclasses.ipynb).

*Last reviewed: November 2026, against Python 3.13.*

## Contents

| section | entries |
| --- | --- |
| [Classes](#classes) | [Attributes and methods](#attributes-and-methods), [Properties](#properties), [Inheritance, `super()` and the MRO](#inheritance-super-and-the-mro) |
| [Special methods](#special-methods) | [Representation, equality, hashing and ordering](#representation-equality-hashing-and-ordering), [Container and callable protocols](#container-and-callable-protocols), [`__slots__`](#__slots__) |
| [dataclasses](#dataclasses) | [`@dataclass`](#dataclass), [dataclass, `NamedTuple`, `TypedDict` or pydantic?](#dataclass-namedtuple-typeddict-or-pydantic) |
| [Enums, ABCs and protocols](#enums-abcs-and-protocols) | [`enum`](#enum), [Abstract base classes and `Protocol`](#abstract-base-classes-and-protocol) |

## Classes

### Attributes and methods

| member | defined as | receives | use |
| --- | --- | --- | --- |
| instance attribute | `self.x = ...` in `__init__` | | per-object state |
| class attribute | `x = ...` in the class body | | constants and shared defaults (never a shared mutable) |
| instance method | `def m(self, ...)` | the instance | behaviour using state |
| class method | `@classmethod def m(cls, ...)` | the class | alternate constructors (`from_dict`, `from_row`) that also work for subclasses |
| static method | `@staticmethod def m(...)` | nothing | a helper that belongs with the class |

**Notes:** assigning `obj.x = ...` creates an instance attribute that **shadows** the class attribute for that object
only. A mutable class attribute (`tags = []`) is shared by every instance, the class-level version of the
mutable-default trap.

In [ ]:
class Table:
    default_format = "parquet"
    shared_bad = []

    def __init__(self, name, columns, fmt=None):
        self.name = name
        self.columns = list(columns)
        self.fmt = fmt or self.default_format

    @classmethod
    def from_dict(cls, d):
        return cls(d["name"], d.get("columns", []), d.get("format"))

    @staticmethod
    def is_valid_name(name):
        return name.isidentifier()

    def describe(self):
        return f"{self.name} ({self.fmt}): {', '.join(self.columns)}"

t = Table.from_dict({"name": "orders", "columns": ["id", "amount"]})
u = Table("users", ["id"], fmt="delta")
t.shared_bad.append("oops")
print(t.describe(), "|", u.describe(), "|", Table.is_valid_name("order-items"), "| shared:", u.shared_bad)

### Properties

`@property` exposes a method as a read-only attribute, computed on access. Add `@x.setter` to validate assignments.
It lets you start with a plain attribute and add logic later without changing callers.
`functools.cached_property` computes once per instance and stores the result.

In [ ]:
from functools import cached_property

class Order:
    def __init__(self, amount, tax_rate=0.2):
        self.amount = amount
        self.tax_rate = tax_rate

    @property
    def amount(self):
        return self._amount

    @amount.setter
    def amount(self, value):
        if value < 0:
            raise ValueError("amount must be non-negative")
        self._amount = value

    @property
    def total(self):
        return round(self.amount * (1 + self.tax_rate), 2)

    @cached_property
    def expensive_profile(self):
        print("  computing once")
        return {"bucket": "large" if self.amount > 100 else "small"}

o = Order(120)
print(o.total, o.expensive_profile, o.expensive_profile)
try:
    o.amount = -5
except ValueError as e:
    print("ValueError:", e)

### Inheritance, `super()` and the MRO

A subclass inherits attributes and methods and can override them. `super().method(...)` calls the next class in the
**method resolution order** (`Cls.__mro__`), which with multiple inheritance is not necessarily the direct parent;
that is what makes cooperative mixins work. `isinstance(obj, Base)` and `issubclass(Sub, Base)` respect inheritance.

**Notes:** prefer composition (holding a helper object) to deep hierarchies. Mixins should be small, call
`super()`, and take `**kwargs` they pass on.

In [ ]:
class Reader:
    def __init__(self, path):
        self.path = path
    def read(self):
        return [f"raw:{self.path}"]

class LoggingMixin:
    def read(self):
        rows = super().read()
        print(f"  read {len(rows)} rows from {self.path}")
        return rows

class CsvReader(LoggingMixin, Reader):
    def read(self):
        return [r.upper() for r in super().read()]

print(CsvReader("a.csv").read(), [c.__name__ for c in CsvReader.__mro__])
print(isinstance(CsvReader("x"), Reader), issubclass(CsvReader, LoggingMixin))

## Special methods

### Representation, equality, hashing and ordering

| method | used by | notes |
| --- | --- | --- |
| `__repr__` | `repr()`, the REPL, containers, logs | unambiguous, ideally `Name(field=...)` |
| `__str__` | `str()`, `print`, f-strings | readable; defaults to `__repr__` |
| `__eq__` | `==` | return `NotImplemented` for unknown types so Python tries the other side |
| `__hash__` | sets, dict keys | defining `__eq__` sets `__hash__ = None` (unhashable) unless you define it; equal objects must hash equal; hash only immutable fields |
| `__lt__` etc. | `<`, `sorted`, `min`, `max` | `functools.total_ordering` fills in the rest from `__eq__` and one of them |

In [ ]:
from functools import total_ordering

@total_ordering
class Version:
    def __init__(self, text):
        self.parts = tuple(int(p) for p in text.split("."))
    def __repr__(self):
        return f"Version({'.'.join(map(str, self.parts))!r})"
    def __eq__(self, other):
        if not isinstance(other, Version):
            return NotImplemented
        return self.parts == other.parts
    def __hash__(self):
        return hash(self.parts)
    def __lt__(self, other):
        if not isinstance(other, Version):
            return NotImplemented
        return self.parts < other.parts

vs = [Version("1.10.0"), Version("1.2.3"), Version("1.2.3")]
print(sorted(vs), max(vs), len(set(vs)), Version("2.0") >= Version("1.99"), Version("1.0") == "1.0")

### Container and callable protocols

| method | enables |
| --- | --- |
| `__len__` | `len(obj)`, and truthiness when `__bool__` is missing |
| `__iter__` | `for`, `list()`, unpacking (often written as a generator with `yield`) |
| `__contains__` | `in` (falls back to iterating) |
| `__getitem__` | `obj[key]`, slicing (receives a `slice`) |
| `__call__` | calling the object like a function (stateful callables) |
| `__enter__` / `__exit__` | `with` blocks (see [errors, logging and testing](python-07-errors-logging-and-testing.ipynb)) |
| `__add__`, `__mul__`, ... | operators; `__radd__` makes `sum()` work from 0 |

In [ ]:
class Batch:
    def __init__(self, rows):
        self._rows = list(rows)
    def __len__(self):
        return len(self._rows)
    def __iter__(self):
        yield from self._rows
    def __contains__(self, row):
        return row in self._rows
    def __getitem__(self, index):
        result = self._rows[index]
        return Batch(result) if isinstance(index, slice) else result
    def __add__(self, other):
        return Batch(self._rows + list(other))
    def __repr__(self):
        return f"Batch({self._rows!r})"

class RateLimiter:
    def __init__(self, limit):
        self.limit, self.calls = limit, 0
    def __call__(self):
        self.calls += 1
        return self.calls <= self.limit

b1 = Batch([1, 2, 3])
print(len(b1), list(b1), 2 in b1, b1[0], b1[1:], b1 + [4], bool(Batch([])))
allow = RateLimiter(2)
print([allow() for _ in range(4)])

### `__slots__`

`__slots__ = ("a", "b")` replaces the per-instance `__dict__` with fixed storage: less memory per object (useful for
millions of small records) and a typo in an attribute name raises `AttributeError` instead of silently creating a new
attribute. Dataclasses take `slots=True`.

In [ ]:
import sys

class Plain:
    def __init__(self, x, y):
        self.x, self.y = x, y

class Slotted:
    __slots__ = ("x", "y")
    def __init__(self, x, y):
        self.x, self.y = x, y

p, s = Plain(1, 2), Slotted(1, 2)
print(sys.getsizeof(p) + sys.getsizeof(p.__dict__), "bytes vs", sys.getsizeof(s), "bytes")
try:
    s.z = 3
except AttributeError as e:
    print("AttributeError:", e)

## dataclasses

### `@dataclass`

`@dataclass` generates `__init__`, `__repr__` and `__eq__` from annotated fields.

| option | effect |
| --- | --- |
| `frozen=True` | instances are immutable (assignment raises `FrozenInstanceError`) and hashable |
| `order=True` | generates `<`, `<=`, ... comparing fields as a tuple, in definition order |
| `slots=True` (3.10+) | uses `__slots__` |
| `kw_only=True` (3.10+) | all fields keyword-only in `__init__` |
| `field(default_factory=list)` | a fresh mutable default per instance (plain `= []` is rejected) |
| `field(repr=False, compare=False)` | exclude a field from repr / equality |
| `__post_init__(self)` | validation and derived fields after `__init__` |

| helper | returns |
| --- | --- |
| `asdict(obj)` / `astuple(obj)` | a (deep) dict / tuple |
| `replace(obj, **changes)` | a copy with fields changed (the way to "modify" frozen instances) |
| `fields(cls)` | field metadata |

In [ ]:
from dataclasses import FrozenInstanceError, asdict, dataclass, field, fields, replace

@dataclass(frozen=True, order=True, slots=True)
class OrderLine:
    order_id: int
    sku: str
    quantity: int = 1
    tags: tuple[str, ...] = ()

    def __post_init__(self):
        if self.quantity <= 0:
            raise ValueError(f"quantity must be positive, got {self.quantity}")

@dataclass
class Batch2:
    lines: list[OrderLine] = field(default_factory=list)
    source: str = field(default="api", compare=False)

a = OrderLine(7, "SKU-1", 2)
b2 = replace(a, quantity=5)
print(a, b2 > a, asdict(a), [f.name for f in fields(OrderLine)])
print(len({a, OrderLine(7, "SKU-1", 2)}), Batch2() == Batch2(source="file"), Batch2().lines is not Batch2().lines)
for bad in (lambda: setattr(a, "quantity", 9), lambda: OrderLine(1, "x", 0)):
    try:
        bad()
    except (FrozenInstanceError, ValueError) as e:
        print(type(e).__name__, e)

### dataclass, `NamedTuple`, `TypedDict` or pydantic?

| choose | when |
| --- | --- |
| `@dataclass` | internal records with behaviour, defaults, validation in `__post_init__` |
| `typing.NamedTuple` | small immutable records that should also be tuples (unpacking, low memory) |
| `TypedDict` | the data must stay a dict (JSON in and out) and you only want type checking |
| pydantic / attrs | parsing and validating untrusted input with type coercion and good errors |

In [ ]:
from typing import NamedTuple

class Point(NamedTuple):
    x: float
    y: float = 0.0

pt = Point(1.5)
x, y = pt
print(pt, x, y, pt._replace(y=2.0), pt == (1.5, 0.0))

## Enums, ABCs and protocols

### `enum`

`Enum` defines a closed set of named constants. `StrEnum` (3.11+) and `IntEnum` members also behave as `str` / `int`,
which is convenient for JSON and databases.

| expression | returns |
| --- | --- |
| `Status.PAID` / `Status["PAID"]` / `Status("paid")` | a member by attribute, name or value (`ValueError` / `KeyError` when unknown) |
| `member.name` / `member.value` | its name / value |
| `list(Status)` | members in definition order |
| `auto()` | an automatic value (lower-cased name for `StrEnum`) |

In [ ]:
from enum import Enum, IntEnum, StrEnum, auto

class Status(StrEnum):
    PLACED = auto()
    PAID = auto()
    SHIPPED = auto()

class Priority(IntEnum):
    LOW = 1
    HIGH = 3

class Color(Enum):
    RED = "r"

print(Status("paid"), Status.PAID == "paid", Status["SHIPPED"].value, [s.name for s in Status], Priority.HIGH > Priority.LOW, Color.RED == "r")
try:
    Status("refunded")
except ValueError as e:
    print("ValueError:", e)

### Abstract base classes and `Protocol`

| tool | checks | use |
| --- | --- | --- |
| `abc.ABC` + `@abstractmethod` | at instantiation: a subclass missing an abstract method raises `TypeError` | a base class you own that subclasses must complete |
| `typing.Protocol` | statically (mypy/pyright); with `@runtime_checkable`, `isinstance` checks method **names** only | describing what a function needs from its argument (duck typing) without requiring inheritance |

In [ ]:
from abc import ABC, abstractmethod
from typing import Protocol, runtime_checkable

class Sink(ABC):
    @abstractmethod
    def write(self, rows): ...
    def write_all(self, batches):
        return sum(self.write(b) for b in batches)

class MemorySink(Sink):
    def __init__(self):
        self.rows = []
    def write(self, rows):
        self.rows.extend(rows)
        return len(rows)

class Broken(Sink):
    pass

print(MemorySink().write_all([[1, 2], [3]]))
try:
    Broken()
except TypeError as e:
    print("TypeError:", e)

@runtime_checkable
class Closeable(Protocol):
    def close(self) -> None: ...

import io
print(isinstance(io.StringIO(), Closeable), isinstance(42, Closeable))

**Next:** practise these entries in [Python practice 7: classes, dataclasses and protocols](../practice/python-practice-07-classes-and-dataclasses.ipynb).